# 01 — Exploratory Data Analysis (PlantVillage, color)

Goal: understand the dataset before any modelling. Because the dataset is
**imbalanced at both hierarchy levels**, the central output of this notebook is a
quantified picture of that imbalance — it justifies the evaluation strategy
(macro/weighted F1, balanced accuracy; **never accuracy as a primary metric**).

Sections:
1. Setup & load the persisted split
2. Class distribution — Level 1 (species) and Level 2 (38 classes)
3. Imbalance ratios per level
4. Sample image grid per class
5. Image dimensions
6. Per-channel mean/std & color histograms
7. Split sanity check (distribution preserved across train/val/test)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Make `src` importable from the notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.splits import load_split, split_summary, LabelMaps  # noqa: E402

sns.set_theme(style="whitegrid")

# Edit if your raw data lives elsewhere.
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "plantvillage dataset" / "color"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT   :", DATA_ROOT, "(exists:", DATA_ROOT.exists(), ")")

In [ ]:
# Load the persisted, labelled split. (Run scripts/prepare_splits.py first.)
df = load_split(SPLITS_DIR, "all")
maps = LabelMaps.from_json(SPLITS_DIR / "label_maps.json")

print(f"Images : {len(df):,}")
print(f"Species (L1): {maps.num_species}  |  Classes (L2): {maps.num_classes}")
df.head()

## 2. Class distribution

In [ ]:
# Level 1 — species
species_counts = df["species"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=species_counts.values, y=species_counts.index, ax=ax, color="steelblue")
ax.set(title="Level 1 — images per species", xlabel="count", ylabel="species")
plt.tight_layout()
plt.show()
species_counts

In [ ]:
# Level 2 — full 38 (species, disease) classes
class_counts = df["class_name"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 11))
sns.barplot(x=class_counts.values, y=class_counts.index, ax=ax, color="indianred")
ax.set(title="Level 2 — images per (species, disease) class", xlabel="count", ylabel="class")
plt.tight_layout()
plt.show()
class_counts

## 3. Imbalance ratios

The imbalance ratio (max support / min support) is the headline number that
justifies macro F1 over accuracy. Reported at both levels, plus per-species
(the imbalance the per-species disease classifiers actually face).

In [ ]:
def imbalance_ratio(counts: pd.Series) -> float:
    return counts.max() / counts.min()

print(f"L1 (species) imbalance ratio : {imbalance_ratio(species_counts):.1f}x")
print(f"L2 (38-class) imbalance ratio: {imbalance_ratio(class_counts):.1f}x")

print("\nPer-species disease imbalance (within each species):")
for species in sorted(df["species"].unique()):
    sub = df.loc[df["species"] == species, "disease"].value_counts()
    ratio = imbalance_ratio(sub) if len(sub) > 1 else 1.0
    print(f"  {species:<28} {len(sub):>2} diseases  ratio={ratio:>5.1f}x  (n={sub.sum():,})")

## 4. Sample images per class

In [ ]:
# One representative image per class. Highlights the near-uniform per-class
# background that drives the known PlantVillage background-bias problem.
classes = sorted(df["class_name"].unique())
n = len(classes)
cols = 6
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.6))
for ax, cls in zip(axes.ravel(), classes):
    sample = df[df["class_name"] == cls].iloc[0]
    img = Image.open(DATA_ROOT / sample["filepath"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(cls.replace("___", "\n"), fontsize=7)
    ax.axis("off")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Image dimensions

In [ ]:
# Sample a subset for speed; PlantVillage color is expected to be uniformly 256x256.
sample_df = df.sample(min(2000, len(df)), random_state=42)
dims = []
for fp in sample_df["filepath"]:
    with Image.open(DATA_ROOT / fp) as im:
        dims.append(im.size)  # (width, height)
dims = pd.DataFrame(dims, columns=["width", "height"])
print(dims.describe())
print("\nUnique (w, h) sizes:", sorted(set(map(tuple, dims.values))))

## 6. Per-channel mean/std & color histograms

Compute on the **train split only** — these stats feed normalization and must
not leak val/test information.

In [ ]:
train_df = df[df["split"] == "train"]
sample_train = train_df.sample(min(2000, len(train_df)), random_state=42)

psum = np.zeros(3)
psum_sq = np.zeros(3)
npix = 0
hist = np.zeros((3, 256), dtype=np.int64)
for fp in sample_train["filepath"]:
    arr = np.asarray(Image.open(DATA_ROOT / fp).convert("RGB"), dtype=np.float64) / 255.0
    pixels = arr.reshape(-1, 3)
    psum += pixels.sum(axis=0)
    psum_sq += (pixels ** 2).sum(axis=0)
    npix += pixels.shape[0]
    for c in range(3):
        hist[c] += np.bincount((arr[..., c] * 255).astype(int).ravel(), minlength=256)

mean = psum / npix
std = np.sqrt(psum_sq / npix - mean ** 2)
print("Per-channel mean (RGB):", np.round(mean, 4))
print("Per-channel std  (RGB):", np.round(std, 4))
print("(ImageNet ref mean: [0.485, 0.456, 0.406], std: [0.229, 0.224, 0.225])")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for c, color in enumerate(["red", "green", "blue"]):
    ax.plot(hist[c] / hist[c].sum(), color=color, label=color)
ax.set(title="Train-split color histograms (normalized)", xlabel="intensity", ylabel="freq")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Split sanity check

Confirm the stratified split preserved per-class proportions across
train/val/test (~70/15/15 everywhere).

In [ ]:
summary = split_summary(df)
props = summary[["train", "val", "test"]].div(summary["total"], axis=0)
print("Per-split proportion stats (should cluster around 0.70 / 0.15 / 0.15):")
print(props.describe().loc[["mean", "min", "max"]].round(3))
summary.head(10)